In [ ]:
# ===================================================================================
# ENTERPRISE MIGRATION SCRIPT V38.9 (50-CHAR LIMIT + EMERGENCY NAME FALLBACK)
# - FIX: Aggressively truncates Service Name to 50 chars (Safe for all DBs).
# - FIX: Adds "Emergency Fallback" using Item ID if naming still fails.
# - RETAINS: Strict Sharing, Thumbnails, Group Mirroring, Metadata.
# ===================================================================================

import pandas as pd
import time
import csv
import os
import datetime
import urllib3
import shutil
import json
import re
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

# Check for ArcPy
HAS_ARCPY = False
try:
    import arcpy
    HAS_ARCPY = True
    print("   [System] ArcPy detected. Surgical Extraction enabled.")
except ImportError:
    print("   [System] ⚠ ArcPy not found. Plan B will likely fail on Views.")

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- NOTEBOOK-SPECIFIC OVERRIDES ---
THROTTLE_SECONDS = 30

# FULL FLEET LIST - Round 1 - staging
# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_1_feature_services.json")
if os.path.exists(_sidecar_path):
    import json as _json
    MULTI_LAYER_IDS = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(MULTI_LAYER_IDS)} Feature Service IDs from sidecar.")
else:
    MULTI_LAYER_IDS = [
        # Example Souce IDs
        # "04950701a5744339acb47bbc43602295",
        # "057160591b7a4e999cd73a2031bbd15d",
        # "070340a24de64147ab026909fc737a07",
    ]

# =============================================================================
# --- LOGGING & CONNECT --------------------------------------------------------
# =============================================================================
if not os.path.exists(TEMP_DIR): os.makedirs(TEMP_DIR)

STATS = {
    "services_scanned": 0,
    "services_skipped_log": 0,
    "published_new": 0,
    "styles_copied": 0,
    "failures": 0,
}

ALREADY_MIGRATED_IDS = set()
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns:
            ALREADY_MIGRATED_IDS = set(df_log["SourceID"].astype(str).str.strip())
        print(f"Loaded history: {len(ALREADY_MIGRATED_IDS)} items previously migrated.")
    except: pass
else:
    try:
        with open(LOG_FILE, mode="w", newline="") as f:
            csv.writer(f).writerow(["SourceID", "LayerIndex", "TargetID", "Title", "MigratedDate", "Type"])
    except: pass

def write_log_entry(source_id, layer_index, target_id, title):
    try:
        with open(LOG_FILE, mode="a", newline="") as f:
            csv.writer(f).writerow([
                source_id, layer_index, target_id, title, 
                datetime.datetime.now().strftime("%m/%d/%Y %H:%M"), "Feature Service"
            ])
    except: print("   ⚠ Log Write Failed")

print("Connecting...")
try:
    session = requests.Session()
    retry_strategy = Retry(
        total=5, 
        backoff_factor=1, 
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "POST"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)
    
    if not target_gis.users.me: raise Exception("Target login failed.")
    print(f" > Target: {target_gis.users.me.username}")

except Exception as e:
    print(f"❌ CONNECTION FAILURE: {e}")
    exit()

# =============================================================================
# --- HELPER: VERSION-SAFE JSON ------------------------------------------------
# =============================================================================
def _featureset_json(fs):
    tj = getattr(fs, "to_json", None)
    if tj is None: raise Exception("FeatureSet missing to_json")
    return tj() if callable(tj) else tj

# =============================================================================
# --- HELPER: FOLDER & OWNER UTILS ---------------------------------------------
# =============================================================================
def get_source_folder_name(source_item):
    """Retrieve the folder name of the source item."""
    try:
        if not source_item.ownerFolder: return None
        user = source_gis.users.get(source_item.owner)
        if user:
            for f in user.folders:
                if f['id'] == source_item.ownerFolder: return f['title']
    except: pass
    return None

def ensure_target_folder_exists(username, folder_name):
    """Ensure a folder exists for a specific user in Target."""
    try:
        user = target_gis.users.get(username)
        if not user: return False
        
        # Check existing folders
        existing_folders = [f['title'] for f in user.folders]
        if folder_name in existing_folders: return True
        
        # Create if missing
        print(f"      + Creating folder '{folder_name}' for user '{username}'...")
        target_gis.content.create_folder(folder_name, owner=username)
        return True
    except Exception as e: 
        print(f"      ⚠ Folder creation error: {e}")
        return False

# NEW HELPER: COPY THUMBNAIL
def copy_thumbnail(source_item, target_item):
    try:
        print("      🖼️ Copying Thumbnail...")
        temp_dir = "thumbnails_temp"
        os.makedirs(temp_dir, exist_ok=True)
        thumb_path = source_item.download_thumbnail(save_folder=temp_dir)
        if thumb_path: 
            target_item.update(thumbnail=thumb_path)
    except: pass

# =============================================================================
# --- HELPER: MIRROR SHARING (STRICT SOURCE MATCH) -----------------------------
# =============================================================================
def mirror_source_sharing(source_item, target_item):
    """
    1. Checks Source Sharing (Public/Org/Private).
    2. STRICTLY applies the same level to Target (No forcing Org).
    3. Maps Source Groups -> Target Groups (by exact Title).
    """
    try:
        print("      👥 Mirroring Sharing Permissions (Source -> Target)...")
        
        # 1. Access Level
        is_public = False
        is_org = False
        
        if source_item.access == 'public':
            is_public = True
            is_org = True # Public implies Org visibility usually
        elif source_item.access == 'org':
            is_org = True
        # Else: Private (Both remain False)
            
        # 2. Get List of Groups (Robust Dictionary Check)
        source_group_list = []
        raw_shared = source_item.shared_with
        
        if isinstance(raw_shared, dict) and 'groups' in raw_shared:
            source_group_list = raw_shared['groups']
        elif isinstance(raw_shared, list):
            source_group_list = raw_shared
            
        target_group_ids = []
        
        if source_group_list:
            print(f"         - Found {len(source_group_list)} source groups. Mapping...")
            for sg in source_group_list:
                # Handle Group Object vs String Name
                sg_title = sg.title if hasattr(sg, 'title') else str(sg)

                # Search for group with exact title in Target
                found_groups = target_gis.groups.search(f"title:\"{sg_title}\"")
                
                # Filter for exact match
                matched_group = next((g for g in found_groups if g.title == sg_title), None)
                
                if matched_group:
                    target_group_ids.append(matched_group.id)
                    print(f"           + Mapped: '{sg_title}'")
                else:
                    print(f"           - Skipped: '{sg_title}' (Not found in Target)")
        
        # 3. Apply
        target_item.share(everyone=is_public, org=is_org, groups=target_group_ids)
        print(f"         ✅ Shared (Org={is_org}, Public={is_public}, Groups={len(target_group_ids)})")

    except Exception as e:
        print(f"      ⚠ Sharing Mirror Failed: {e}")

# =============================================================================
# --- HELPER: ROBUST SQL EXTRACTION --------------------------------------------
# =============================================================================
def extract_data_by_sql(source_item, save_folder):
    if not HAS_ARCPY: return None
    try:
        gdb_name = f"LocalExt_{source_item.id}.gdb"
        gdb_path = os.path.join(save_folder, gdb_name)
        
        if arcpy.Exists(gdb_path): 
            try: arcpy.management.Delete(gdb_path)
            except: pass
        arcpy.management.CreateFileGDB(save_folder, gdb_name)
        print(f"   [Fallback] Created local GDB: {gdb_name}")

        layers_processed = 0
        for i, lyr in enumerate(source_item.layers):
            safe_name = f"Lyr_{i}".replace(" ", "_")
            out_fc = os.path.join(gdb_path, safe_name)
            
            oid_field = "OBJECTID" 
            if hasattr(lyr.properties, "objectIdField"):
                oid_field = lyr.properties.objectIdField
            
            print(f"      > Layer {i}: {lyr.properties.name} (ID Field: {oid_field})")
            
            try:
                oid_res = lyr.query(where="1=1", return_ids_only=True)
                if 'objectIds' in oid_res: all_oids = oid_res['objectIds']
                elif 'ids' in oid_res: all_oids = oid_res['ids']
                else: continue
                all_oids.sort()
                if not all_oids: continue
                print(f"        Found {len(all_oids)} features.")
            except: continue

            created = False
            oid_chunks = [all_oids[x:x+ID_BATCH_SIZE] for x in range(0, len(all_oids), ID_BATCH_SIZE)]
            
            for idx, chunk in enumerate(oid_chunks):
                json_file = os.path.join(save_folder, f"temp_{source_item.id}_L{i}_B{idx}.json")
                oid_list_str = ",".join(map(str, chunk))
                where_clause = f"{oid_field} IN ({oid_list_str})"
                
                success = False
                for attempt in range(1, 11):
                    try:
                        print(f"        - Fetching batch {idx+1}/{len(oid_chunks)} ({len(chunk)} rows)...", end="\r")
                        fs = lyr.query(
                            where=where_clause,
                            out_fields="*", 
                            return_geometry=True,
                            order_by_fields=oid_field
                        )
                        if fs is None or not hasattr(fs, 'features'): raise Exception("Empty response")
                        with open(json_file, 'w', encoding='utf-8') as f:
                            f.write(_featureset_json(fs))
                        
                        if not created:
                            arcpy.conversion.JSONToFeatures(json_file, out_fc)
                            created = True
                        else:
                            temp_fc = os.path.join(gdb_path, "temp_append")
                            if arcpy.Exists(temp_fc): arcpy.management.Delete(temp_fc)
                            arcpy.conversion.JSONToFeatures(json_file, temp_fc)
                            arcpy.management.Append(temp_fc, out_fc, schema_type="NO_TEST")
                            arcpy.management.Delete(temp_fc)
                        success = True
                        break
                    except Exception as chunk_err:
                        time.sleep(2 ** attempt)
                
                if os.path.exists(json_file): os.remove(json_file)
                if not success: print(f"\n        ❌ Batch {idx+1} failed HARD.")

            print(f"        ✅ Layer {i} complete.                                 ")
            layers_processed += 1

        if layers_processed == 0: return None

        print("   [System] Clearing locks...")
        arcpy.ClearWorkspaceCache_management()
        time.sleep(5) 
        
        # Zip FOLDER (Correct Structure)
        zip_base = gdb_path
        root_dir = os.path.dirname(gdb_path)
        base_dir = os.path.basename(gdb_path)
        shutil.make_archive(zip_base, 'zip', root_dir, base_dir)
        return gdb_path + ".zip"

    except Exception as e:
        print(f"   ❌ Extraction Error: {e}")
        return None

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print("\nStarting V38.9 Migration (50-Char Limit + Emergency Fallback + Fixes)...")
start_time = time.time()

for source_id in MULTI_LAYER_IDS:
    try:
        STATS["services_scanned"] += 1
        # CHECK 1: HISTORY LOG
        if str(source_id) in ALREADY_MIGRATED_IDS:
            print(f"\n[Skip] {source_id} found in log.")
            STATS["services_skipped_log"] += 1
            continue

        source_item = source_gis.content.get(source_id)
        if not source_item:
            print(f"❌ Source {source_id} not found.")
            STATS["failures"] += 1
            continue

        print(f"\nProcessing: {source_item.title} ({source_id})...")
        export_name = f"Migrate_{source_id}_{int(time.time())}"
        local_zip_path = None
        fgdb_item = None
        
        # --- DETERMINE TARGET TITLE ---
        target_title = source_item.title
        if APPEND_MIGRATED:
            target_title = f"{source_item.title} (Migrated)"

        # --- PLAN A: SERVER EXPORT ---
        print(" > 1. Attempting Server Export...")
        try:
            fgdb_item = source_item.export(title=export_name, export_format="File Geodatabase")
            local_zip_path = fgdb_item.download(save_path=TEMP_DIR)
            print("   ✅ Plan A Successful.")
        except Exception as e:
            print(f"   ⚠ Plan A Failed. Switching to Plan B (Bulldozer)...")
            local_zip_path = extract_data_by_sql(source_item, TEMP_DIR)
            
            if not local_zip_path or not os.path.exists(local_zip_path):
                print(f"   ❌ Plan B failed or file missing: {local_zip_path}")
                STATS["failures"] += 1
                continue
        finally:
            if fgdb_item: 
                try: fgdb_item.delete()
                except: pass

        # --- PRE-PUBLISH CHECK (NO DELETE) ---
        print(" > 2. Checking Target for Conflicts...")
        existing = []
        for attempt in range(1, 4):
            try:
                # Search by Title OR 'name' (Service Name)
                # Note: We search for the *calculated* target title now
                query = f"(title:\"{target_title}\" OR name:\"{target_title}\")"
                existing = target_gis.content.search(query, item_type="Feature Service")
                break 
            except Exception as search_err:
                print(f"      ⚠ Search connection failed (Try {attempt}/3). Waiting...")
                time.sleep(10)

        # CHECK 2: LIVE PORTAL CHECK
        for exist_item in existing:
            if exist_item.title == target_title or exist_item.name == target_title:
                print(f"   ⚠ Found existing service '{exist_item.title}' in Target.")
                print(f"      (Auto-delete is DISABLED. If you didn't delete this manually, publishing may fail!)")
                # NOTE: exist_item.delete() REMOVED per user request

        # --- UPLOAD & PUBLISH ---
        print(" > 3. Uploading & Publishing...")
        
        # --- FIX: DELETE EXISTING STALE ITEM IF EXISTS ---
        stale_items = target_gis.content.search(f"title:{os.path.basename(local_zip_path)}", item_type="File Geodatabase")
        for stale in stale_items:
            try:
                print(f"      🗑️ Deleting stale upload item: {stale.title}...")
                stale.delete()
            except: pass

        fgdb_props = {
            'title': target_title, 
            'type': 'File Geodatabase', 
            'tags': source_item.tags,
            'snippet': source_item.snippet, 
            'description': source_item.description, # Added description
            'accessInformation': source_item.accessInformation,
            'licenseInfo': source_item.licenseInfo  # Added Terms of Use
        }
        target_fgdb = target_gis.content.add(item_properties=fgdb_props, data=local_zip_path)
        
        new_item = None
        try:
            # --- FIX: STRICT 50-CHAR TRUNCATION (V38.9) ---
            clean_name = re.sub(r'[^a-zA-Z0-9_]', '_', target_title)
            clean_name = re.sub(r'_+', '_', clean_name).strip('_')
            
            if clean_name and clean_name[0].isdigit():
                clean_name = "S_" + clean_name
            
            # TRUNCATE TO 50
            if len(clean_name) > 50:
                print(f"      ⚠ Name too long ({len(clean_name)} chars). Truncating to 50.")
                clean_name = clean_name[:50]
                clean_name = clean_name.strip('_')

            print(f"      (Sanitized Service Name: {clean_name})")
            
            new_item = target_fgdb.publish(publish_parameters={"name": clean_name})
        except Exception as pub_err:
            print(f"      ⚠ Primary Publish Failed: {pub_err}")
            
            # --- EMERGENCY FALLBACK ---
            # If standard naming fails, use a completely safe, generic name based on ID
            try:
                emergency_name = f"Service_{source_id}_Migrated"
                print(f"      🚨 Attempting Emergency Publish as: {emergency_name}")
                new_item = target_fgdb.publish(publish_parameters={"name": emergency_name})
            except Exception as final_err:
                raise final_err # Give up if even this fails
        
        if new_item:
            STATS["published_new"] += 1
            print(f"   ✅ Created Target ID: {new_item.id} (Title: {new_item.title})")
            ALREADY_MIGRATED_IDS.add(str(source_id))
        else:
            raise Exception("Publishing failed.")
        
        # --- FIX: SHARING MUST HAPPEN BEFORE REASSIGNING OWNER ---
        mirror_source_sharing(source_item, new_item)

        # --- OWNER & FOLDER ASSIGNMENT ---
        print(" > 4. Assigning Owner & Folder...")
        try:
            source_owner = source_item.owner
            target_owner_to_use = DEFAULT_OWNER
            target_folder_to_use = DEFAULT_FOLDER

            # Check if Source Owner exists in Target
            found_user = target_gis.users.get(source_owner)
            
            if found_user:
                print(f"      👤 Source owner '{source_owner}' found in target.")
                target_owner_to_use = source_owner
                
                # Try to get source folder name
                src_folder_name = get_source_folder_name(source_item)
                if src_folder_name:
                    if ensure_target_folder_exists(target_owner_to_use, src_folder_name):
                        target_folder_to_use = src_folder_name
                    else:
                        print(f"      ⚠ Could not create folder '{src_folder_name}'. Using Default.")
                else:
                    target_folder_to_use = None # Root
            else:
                print(f"      👤 Source owner '{source_owner}' NOT found. Defaulting to '{DEFAULT_OWNER}'.")
                # Ensure default folder exists for default user
                ensure_target_folder_exists(DEFAULT_OWNER, DEFAULT_FOLDER)

            # Reassign
            print(f"      📂 Moving to: Owner={target_owner_to_use}, Folder={target_folder_to_use}")
            new_item.reassign_to(target_owner_to_use, target_folder=target_folder_to_use)

        except Exception as owner_err:
            print(f"      ⚠ Ownership assignment failed: {owner_err}")

        # --- FINALIZING ---
        print(" > 5. Finalizing (Metadata, Styles, Scale, Extent)...")
        
        # FIX 1: COPY EXTENT
        try:
            if source_item.extent:
                new_item.update(item_properties={
                    'extent': source_item.extent,
                    'description': source_item.description,
                    'snippet': source_item.snippet,
                    'accessInformation': source_item.accessInformation,
                    'licenseInfo': source_item.licenseInfo
                })
        except: pass

        # NEW: COPY THUMBNAIL
        copy_thumbnail(source_item, new_item)

        try:
            for i, src_lyr in enumerate(source_item.layers):
                if i < len(new_item.layers):
                    tgt_lyr = new_item.layers[i]
                    original_name = src_lyr.properties.name
                    print(f"      > Layer {i}: {original_name} (Updating Props & Scale)")
                    
                    # FIX 2: COPY SCALE DEPENDENCY
                    update_dict = {
                        'name': original_name,
                        'minScale': src_lyr.properties.minScale,
                        'maxScale': src_lyr.properties.maxScale
                    }
                    if hasattr(src_lyr.properties, 'copyrightText'):
                        update_dict['copyrightText'] = src_lyr.properties.copyrightText
                    elif source_item.accessInformation: 
                        update_dict['copyrightText'] = source_item.accessInformation

                    if hasattr(src_lyr.properties, 'drawingInfo'):
                         update_dict['drawingInfo'] = dict(src_lyr.properties.drawingInfo)
                         STATS["styles_copied"] += 1
                    
                    tgt_lyr.manager.update_definition(update_dict)
        except Exception as name_err:
            print(f"   ⚠ Name/Style Update Warning: {name_err}")

        # Logging (Uses the NEW title with suffix)
        write_log_entry(source_id, "Service", new_item.id, new_item.title)
        try:
            for lyr in new_item.layers:
                write_log_entry(source_id, lyr.properties.id, new_item.id, lyr.properties.name)
        except: pass

        try:
            target_fgdb.delete()
            if os.path.exists(local_zip_path): os.remove(local_zip_path)
            raw_gdb = local_zip_path.replace(".zip", "")
            if os.path.exists(raw_gdb): shutil.rmtree(raw_gdb, ignore_errors=True)
        except: pass
        
        print(f"   💤 Cooling down for {THROTTLE_SECONDS} seconds...")
        time.sleep(THROTTLE_SECONDS)

    except Exception as e:
        print(f"❌ FAILED on {source_id}: {e}")
        STATS["failures"] += 1

# =============================================================================
# --- REPORT -------------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "=" * 60)
print("      MASTER MIGRATION REPORT (V38.9)      ")
print("=" * 60)
print(f" ⏱️  Total Duration:             {duration_str}")
print("-" * 60)
print(f" 📦 Services Processed:        {STATS['services_scanned']}")
print(f" 🧾 Skipped (log):             {STATS['services_skipped_log']}")
print(f" 🚀 New Services Published:    {STATS['published_new']}")
print(f" 🎨 Styles/Scale Copied:       {STATS['styles_copied']}")
print(f" ❌ Failures:                  {STATS['failures']}")
print("=" * 60)